<!-- AUTO-GENERATED from notebooks/02_eda_phase0.ipynb — do not edit on disk. Run: python scripts/sync_kaggle_eda.py --push -->
## Kaggle bootstrap

Installs this repo on Kaggle. Skipped when running locally.

Repo: `https://github.com/sh0ch/RSNA-knee-abnormality-detection.git`

Sync from your machine: `python scripts/sync_kaggle_eda.py --push`


In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/sh0ch/RSNA-knee-abnormality-detection.git"
BRANCH = "main"
WORK_DIR = "/kaggle/working/rsna_knee_repo"

ON_KAGGLE = (
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
    or "KAGGLE_URL_BASE" in os.environ
    or os.path.isdir("/kaggle/working")
)

if ON_KAGGLE:
    if not os.path.exists(WORK_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, WORK_DIR],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", WORK_DIR, "pull", "--ff-only", "origin", BRANCH],
            check=False,
        )
    subprocess.run(["pip", "install", "-q", "-e", f"{WORK_DIR}[dev]"], check=True)
    sys.path.insert(0, f"{WORK_DIR}/src")
else:
    print("Local run — skipping Kaggle bootstrap.")


# Phase 0 — EDA (step by step)

Exploratory analysis for [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection).

We build this notebook incrementally. **Step 1:** load raw files as they ship from the competition — no column renaming, no rescaling, no derived features.

**Run on Kaggle** for real data. Locally: `python scripts/create_sample_data.py` first.

Confirmed findings go in **[docs/PROJECT_LOG.md](../docs/PROJECT_LOG.md)**.

Sync to Kaggle: `python scripts/sync_kaggle_eda.py --push`

## Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import pydicom

# Kaggle: repo cloned above; local: search from notebooks/
WORK_DIR = Path("/kaggle/working/rsna_knee_repo")
if WORK_DIR.is_dir():
    REPO_ROOT = WORK_DIR
else:
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "src" / "rsna_knee").is_dir():
            REPO_ROOT = candidate
            break
    else:
        REPO_ROOT = Path.cwd().parent

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from rsna_knee.constants import (
    SAMPLE_SUBMISSION_CSV,
    TEST_CSV,
    TEST_SERIES_CSV,
    TRAIN_CSV,
    TRAIN_SERIES_CSV,
    TRAIN_SERIES_DIR,
)
from rsna_knee.data.schema import read_competition_csv
from rsna_knee.utils.paths import default_data_root, is_kaggle_kernel

DATA_ROOT = default_data_root()
print(f"Environment : {'Kaggle' if is_kaggle_kernel() else 'local'}")
print(f"Data root   : {DATA_ROOT}")

## Step 1 — Raw CSV tables

Read competition CSVs exactly as stored on disk. Column names and dtypes are unchanged.

In [ ]:
csv_files = {
    "train": TRAIN_CSV,
    "train_series": TRAIN_SERIES_CSV,
    "test": TEST_CSV,
    "test_series": TEST_SERIES_CSV,
    "sample_submission": SAMPLE_SUBMISSION_CSV,
}

raw_tables: dict[str, pd.DataFrame] = {}
for name, filename in csv_files.items():
    path = DATA_ROOT / filename
    raw_tables[name] = read_competition_csv(path)
    print(f"{name:18s}  {path.name:22s}  rows={len(raw_tables[name]):,}  cols={len(raw_tables[name].columns)}")

In [ ]:
for name, df in raw_tables.items():
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    print("Columns:", list(df.columns))
    display(df.head(3))
    df.info()

## Step 2 — Raw DICOM files

Peek at one series on disk: file layout, metadata tags, and unstretched pixel values.

In [ ]:
series_root = DATA_ROOT / TRAIN_SERIES_DIR
study_dirs = sorted(p for p in series_root.iterdir() if p.is_dir())
print(f"Studies under {series_root}: {len(study_dirs)}")

sample_study = study_dirs[0]
series_dirs = sorted(p for p in sample_study.iterdir() if p.is_dir())
sample_series = series_dirs[0]
dcm_files = sorted(sample_series.glob("*.dcm"))

print(f"Sample study : {sample_study.name}")
print(f"Sample series: {sample_series.name}")
print(f"DICOM files  : {len(dcm_files)} (*.dcm in series folder)")
print("First 5 files:")
for path in dcm_files[:5]:
    print(f"  {path.name}")

In [ ]:
ds = pydicom.dcmread(str(dcm_files[0]))

print("Key DICOM tags (first slice):")
for tag in (
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SOPInstanceUID",
    "InstanceNumber",
    "Modality",
    "Rows",
    "Columns",
    "PixelSpacing",
    "SliceThickness",
    "ImagePositionPatient",
    "RescaleSlope",
    "RescaleIntercept",
    "TransferSyntaxUID",
):
    print(f"  {tag:22s}  {getattr(ds, tag, '<missing>')}")

pixels = ds.pixel_array
print(f"\nRaw pixel_array: shape={pixels.shape}, dtype={pixels.dtype}")
print(f"  min={pixels.min()}, max={pixels.max()}, mean={pixels.mean():.2f}")

### View one slice

Yes — `pixel_array` **is the MRI image** for that slice: a 2D grid of signal intensities.

- One `.dcm` file = **one 2D slice** (shape `Rows × Columns`, e.g. 512×512 on real data).
- All slices in a series folder = **one 3D MRI volume** (stack along the slice axis).
- `Modality: MR` confirms magnetic resonance imaging.
- Values are raw scanner units (often 12–16 bit). We have **not** applied `RescaleSlope` / `RescaleIntercept` yet — that comes later if we want Hounsfield-like or standardized intensities.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Middle slice often shows more anatomy than the first/last
mid_idx = len(dcm_files) // 2
slices_to_show = [0, mid_idx, len(dcm_files) - 1]

for ax, idx in zip(axes, slices_to_show):
    ds_i = pydicom.dcmread(str(dcm_files[idx]))
    img = ds_i.pixel_array
    ax.imshow(img, cmap="gray")
    ax.set_title(f"slice {idx + 1} / {len(dcm_files)}")
    ax.axis("off")

plt.suptitle(f"Raw pixel_array — series {sample_series.name[:20]}…")
plt.tight_layout()
plt.show()